

# Data Analyst Agent for NHS Cancer Data

If the best human data analyst is asked to analyze NHS cancer data, the agent could work like this:

1. **Understand the problem**

   * Define the question, cancer type, patient group, and time period.

2. **Collect and prepare the data**

   * Gather NHS cancer records, clean the data, and combine different datasets.

3. **Analyse the data**

   * Find trends, compare patient groups, and identify key patterns or risks.

4. **Review the findings**

   * Check if the results are accurate and compare them with NHS benchmarks.

5. **Create the final report**

   * Present charts, conclusions, and recommendations for NHS teams.




In [ ]:
# Step 1: Understand the Problem (Google Colab)


# Install required packages
!pip install -q langchain-groq langchain

from langchain_groq import ChatGroq

# 1. Create the LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="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",
    temperature=0
)

# 2. Give the agent the NHS cancer analysis request
question = """
We want to analyse NHS cancer data.

User question:
Which patient groups are more likely to have delayed cancer diagnosis?
"""

# 3. Ask the LLM to convert the question into a clear analysis plan
prompt = f"""
You are an NHS Cancer Data Analyst Agent.

Your job is only to understand the problem.

For the given question:
- Identify the main objective
- Identify which cancer types may be relevant
- Identify which patient groups should be compared
- Identify which data fields will be needed

Question:
{question}
"""

response = llm.invoke(prompt)

# 4. Print the result
print(response.content)



To understand the problem, let's break it down into its key components:

1. **Main Objective**: The main objective is to identify patient groups that are more likely to experience delayed cancer diagnosis.

2. **Relevant Cancer Types**: All cancer types may be relevant, but some specific types might be more prone to delayed diagnosis due to their nature (e.g., pancreatic cancer, ovarian cancer) or due to symptoms that are often mistaken for other less severe conditions. However, without more specific information, it's reasonable to consider all cancer types as potentially relevant.

3. **Patient Groups for Comparison**: The patient groups to be compared could include, but are not limited to:
   - Demographic groups (age, gender, ethnicity)
   - Socioeconomic groups (income level, education level, employment status)
   - Geographic groups (urban vs. rural, specific regions within the country)
   - Groups with different access to healthcare services (insured vs. uninsured, although this 

In [ ]:
# Step 2: Collect and Prepare NHS Cancer Data Using Sample CSV (Google Colab)

# Install required packages
!pip install -q langchain langchain-groq pandas

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
import pandas as pd

# --------------------------------------------------
# 1. Create sample NHS cancer CSV automatically
# --------------------------------------------------
sample_data = {
    "patient_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "age": [45, 67, 52, 71, 39, 58, 64, 50],
    "gender": ["Female", "Male", "Female", "Male", "Female", "Male", "Female", "Male"],
    "ethnicity": [
        "White", "Asian", "Black", "White",
        "Asian", "Black", "White", "Asian"
    ],
    "region": [
        "London", "Manchester", "Birmingham", "Leeds",
        "London", "Liverpool", "Bristol", "Manchester"
    ],
    "cancer_type": [
        "Breast", "Lung", "Breast", "Prostate",
        "Cervical", "Lung", "Breast", "Bowel"
    ],
    "referral_date": [
        "2024-01-10", "2024-02-12", "2024-01-25", "2024-03-01",
        "2024-02-15", "2024-01-18", "2024-02-05", "2024-03-10"
    ],
    "diagnosis_date": [
        "2024-02-05", "2024-03-20", "2024-02-15", "2024-04-10",
        "2024-03-01", "2024-02-25", "2024-03-10", "2024-04-01"
    ]
}

# Save sample data as CSV
sample_df = pd.DataFrame(sample_data)
file_name = "nhs_cancer_data.csv"
sample_df.to_csv(file_name, index=False)

print("Sample CSV created:", file_name)

# --------------------------------------------------
# 2. Create the LLM
# --------------------------------------------------
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="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",
    temperature=0
)

# --------------------------------------------------
# 3. Tool: load dataset info
# --------------------------------------------------
@tool
def load_nhs_cancer_data() -> str:
    """Load the NHS cancer CSV file and return basic dataset information."""
    df = pd.read_csv(file_name)

    return (
        f"Dataset loaded successfully.\n"
        f"Rows: {len(df)}\n"
        f"Columns: {list(df.columns)}"
    )

# --------------------------------------------------
# 4. Tool: clean and prepare dataset
# --------------------------------------------------
@tool
def prepare_nhs_cancer_data() -> str:
    """Clean and prepare the NHS cancer CSV file."""
    df = pd.read_csv(file_name)

    # Clean column names
    df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]

    # Keep only useful columns
    useful_columns = [
        "patient_id",
        "age",
        "gender",
        "ethnicity",
        "region",
        "cancer_type",
        "diagnosis_date",
        "referral_date"
    ]

    df = df[useful_columns]

    # Remove duplicate rows
    df = df.drop_duplicates()

    # Remove rows with missing patient_id
    df = df.dropna(subset=["patient_id"])

    # Convert date columns
    df["diagnosis_date"] = pd.to_datetime(df["diagnosis_date"])
    df["referral_date"] = pd.to_datetime(df["referral_date"])

    # Save cleaned dataset
    cleaned_file_name = "cleaned_nhs_cancer_data.csv"
    df.to_csv(cleaned_file_name, index=False)

    return (
        f"Data prepared successfully.\n"
        f"Cleaned rows: {len(df)}\n"
        f"Cleaned columns: {list(df.columns)}\n"
        f"Saved file: {cleaned_file_name}"
    )

# --------------------------------------------------
# 5. Put tools in a list
# --------------------------------------------------
tools = [load_nhs_cancer_data, prepare_nhs_cancer_data]

# --------------------------------------------------
# 6. Create the agent
# --------------------------------------------------
agent = create_agent(
    model=llm,
    tools=tools
)

# --------------------------------------------------
# 7. Run Step 2
# --------------------------------------------------
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Load the NHS cancer CSV dataset, "
                "prepare the data for analysis, "
                "and tell me the final cleaned rows, cleaned columns, "
                "and saved file name only."
            )
        }
    ]
})

# --------------------------------------------------
# 8. Print the result
# --------------------------------------------------
print(result["messages"][-1].content)



Sample CSV created: nhs_cancer_data.csv
Cleaned rows: 8
Cleaned columns: ['patient_id', 'age', 'gender', 'ethnicity', 'region', 'cancer_type', 'diagnosis_date', 'referral_date']
Saved file: cleaned_nhs_cancer_data.csv


In [ ]:
# Step 3: Analyse NHS Cancer Data
# This version works even if raw and cleaned CSV files do not exist

# Install required packages
!pip install -q langchain langchain-groq pandas

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
import pandas as pd
import os

# --------------------------------------------------
# 1. Create the LLM
# --------------------------------------------------
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="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",
    temperature=0
)

# --------------------------------------------------
# 2. File names
# --------------------------------------------------
raw_file = "nhs_cancer_data.csv"
cleaned_file = "cleaned_nhs_cancer_data.csv"

# --------------------------------------------------
# 3. Create sample raw CSV if missing
# --------------------------------------------------
if not os.path.exists(raw_file):
    sample_data = {
        "patient_id": [1, 2, 3, 4, 5, 6, 7, 8],
        "age": [45, 67, 52, 71, 39, 58, 64, 50],
        "gender": ["Female", "Male", "Female", "Male", "Female", "Male", "Female", "Male"],
        "ethnicity": [
            "White", "Asian", "Black", "White",
            "Asian", "Black", "White", "Asian"
        ],
        "region": [
            "London", "Manchester", "Birmingham", "Leeds",
            "London", "Liverpool", "Bristol", "Manchester"
        ],
        "cancer_type": [
            "Breast", "Lung", "Breast", "Prostate",
            "Cervical", "Lung", "Breast", "Bowel"
        ],
        "referral_date": [
            "2024-01-10", "2024-02-12", "2024-01-25", "2024-03-01",
            "2024-02-15", "2024-01-18", "2024-02-05", "2024-03-10"
        ],
        "diagnosis_date": [
            "2024-02-05", "2024-03-20", "2024-02-15", "2024-04-10",
            "2024-03-01", "2024-02-25", "2024-03-10", "2024-04-01"
        ]
    }

    sample_df = pd.DataFrame(sample_data)
    sample_df.to_csv(raw_file, index=False)
    print(f"Created sample file: {raw_file}")

# --------------------------------------------------
# 4. Create cleaned CSV if missing
# --------------------------------------------------
if not os.path.exists(cleaned_file):
    df = pd.read_csv(raw_file)

    # Clean column names
    df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]

    # Keep useful columns
    useful_columns = [
        "patient_id",
        "age",
        "gender",
        "ethnicity",
        "region",
        "cancer_type",
        "diagnosis_date",
        "referral_date"
    ]
    df = df[useful_columns]

    # Remove duplicates
    df = df.drop_duplicates()

    # Remove rows with missing patient_id
    df = df.dropna(subset=["patient_id"])

    # Convert date columns
    df["diagnosis_date"] = pd.to_datetime(df["diagnosis_date"])
    df["referral_date"] = pd.to_datetime(df["referral_date"])

    # Save cleaned file
    df.to_csv(cleaned_file, index=False)
    print(f"Created cleaned file: {cleaned_file}")

# --------------------------------------------------
# 5. Tool: analyse diagnosis delay
# --------------------------------------------------
@tool
def analyse_diagnosis_delay() -> str:
    """Analyse which gender and region have the highest average diagnosis delay."""

    df = pd.read_csv(cleaned_file)

    # Convert dates
    df["diagnosis_date"] = pd.to_datetime(df["diagnosis_date"])
    df["referral_date"] = pd.to_datetime(df["referral_date"])

    # Create delay column
    df["delay_days"] = (df["diagnosis_date"] - df["referral_date"]).dt.days

    # Average delay by gender
    gender_delay = df.groupby("gender")["delay_days"].mean().sort_values(ascending=False)

    # Average delay by region
    region_delay = df.groupby("region")["delay_days"].mean().sort_values(ascending=False)

    highest_gender = gender_delay.index[0]
    highest_gender_delay = round(gender_delay.iloc[0], 1)

    highest_region = region_delay.index[0]
    highest_region_delay = round(region_delay.iloc[0], 1)

    return (
        f"Gender with highest average delay: {highest_gender} ({highest_gender_delay} days)\n"
        f"Region with highest average delay: {highest_region} ({highest_region_delay} days)"
    )

# --------------------------------------------------
# 6. Put tools in a list
# --------------------------------------------------
tools = [analyse_diagnosis_delay]

# --------------------------------------------------
# 7. Create the agent
# --------------------------------------------------
agent = create_agent(
    model=llm,
    tools=tools
)

# --------------------------------------------------
# 8. Run Step 3
# --------------------------------------------------
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Analyse the NHS cancer dataset and tell me only "
                "which gender and which region have the highest "
                "average delay between referral date and diagnosis date."
            )
        }
    ]
})

# --------------------------------------------------
# 9. Print result
# --------------------------------------------------
print(result["messages"][-1].content)

Created sample file: nhs_cancer_data.csv
Created cleaned file: cleaned_nhs_cancer_data.csv
The gender with the highest average delay between referral date and diagnosis date is Male, and the region with the highest average delay is Leeds.


In [ ]:
# Step 4: Review NHS Cancer Data Findings

# Install required packages
!pip install -q langchain langchain-groq pandas

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
import pandas as pd
import os

# --------------------------------------------------
# 1. Create the LLM
# --------------------------------------------------
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="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",
    temperature=0
)

# --------------------------------------------------
# 2. File names
# --------------------------------------------------
raw_file = "nhs_cancer_data.csv"
cleaned_file = "cleaned_nhs_cancer_data.csv"

# --------------------------------------------------
# 3. Create sample raw CSV if missing
# --------------------------------------------------
if not os.path.exists(raw_file):
    sample_data = {
        "patient_id": [1, 2, 3, 4, 5, 6, 7, 8],
        "age": [45, 67, 52, 71, 39, 58, 64, 50],
        "gender": ["Female", "Male", "Female", "Male", "Female", "Male", "Female", "Male"],
        "ethnicity": [
            "White", "Asian", "Black", "White",
            "Asian", "Black", "White", "Asian"
        ],
        "region": [
            "London", "Manchester", "Birmingham", "Leeds",
            "London", "Liverpool", "Bristol", "Manchester"
        ],
        "cancer_type": [
            "Breast", "Lung", "Breast", "Prostate",
            "Cervical", "Lung", "Breast", "Bowel"
        ],
        "referral_date": [
            "2024-01-10", "2024-02-12", "2024-01-25", "2024-03-01",
            "2024-02-15", "2024-01-18", "2024-02-05", "2024-03-10"
        ],
        "diagnosis_date": [
            "2024-02-05", "2024-03-20", "2024-02-15", "2024-04-10",
            "2024-03-01", "2024-02-25", "2024-03-10", "2024-04-01"
        ]
    }

    sample_df = pd.DataFrame(sample_data)
    sample_df.to_csv(raw_file, index=False)
    print(f"Created sample file: {raw_file}")

# --------------------------------------------------
# 4. Create cleaned CSV if missing
# --------------------------------------------------
if not os.path.exists(cleaned_file):
    df = pd.read_csv(raw_file)

    # Clean column names
    df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]

    # Keep useful columns
    useful_columns = [
        "patient_id",
        "age",
        "gender",
        "ethnicity",
        "region",
        "cancer_type",
        "diagnosis_date",
        "referral_date"
    ]
    df = df[useful_columns]

    # Remove duplicates
    df = df.drop_duplicates()

    # Remove rows with missing patient_id
    df = df.dropna(subset=["patient_id"])

    # Convert date columns
    df["diagnosis_date"] = pd.to_datetime(df["diagnosis_date"])
    df["referral_date"] = pd.to_datetime(df["referral_date"])

    # Save cleaned file
    df.to_csv(cleaned_file, index=False)
    print(f"Created cleaned file: {cleaned_file}")

# --------------------------------------------------
# 5. Tool: review findings
# --------------------------------------------------
@tool
def review_findings() -> str:
    """Review whether the diagnosis delay findings look valid and provide a short conclusion."""

    df = pd.read_csv(cleaned_file)

    # Convert dates
    df["diagnosis_date"] = pd.to_datetime(df["diagnosis_date"])
    df["referral_date"] = pd.to_datetime(df["referral_date"])

    # Create delay column
    df["delay_days"] = (df["diagnosis_date"] - df["referral_date"]).dt.days

    # Basic checks
    total_rows = len(df)
    missing_delay = df["delay_days"].isna().sum()
    negative_delay = (df["delay_days"] < 0).sum()

    # Average delay
    avg_delay = round(df["delay_days"].mean(), 1)

    # Highest average delay by gender
    gender_delay = df.groupby("gender")["delay_days"].mean().sort_values(ascending=False)
    highest_gender = gender_delay.index[0]
    highest_gender_delay = round(gender_delay.iloc[0], 1)

    # Highest average delay by region
    region_delay = df.groupby("region")["delay_days"].mean().sort_values(ascending=False)
    highest_region = region_delay.index[0]
    highest_region_delay = round(region_delay.iloc[0], 1)

    # Simple review status
    if missing_delay == 0 and negative_delay == 0:
        review_status = "Findings look valid for this sample dataset."
    else:
        review_status = "Findings need review because some delay values are missing or invalid."

    return (
        f"Total rows reviewed: {total_rows}\n"
        f"Missing delay values: {missing_delay}\n"
        f"Negative delay values: {negative_delay}\n"
        f"Overall average delay: {avg_delay} days\n"
        f"Highest delay by gender: {highest_gender} ({highest_gender_delay} days)\n"
        f"Highest delay by region: {highest_region} ({highest_region_delay} days)\n"
        f"Review conclusion: {review_status}"
    )

# --------------------------------------------------
# 6. Put tools in a list
# --------------------------------------------------
tools = [review_findings]

# --------------------------------------------------
# 7. Create the agent
# --------------------------------------------------
agent = create_agent(
    model=llm,
    tools=tools
)

# --------------------------------------------------
# 8. Run Step 4
# --------------------------------------------------
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Review the NHS cancer analysis findings and tell me "
                "if the results look valid. Also give the highest delay "
                "by gender and region."
            )
        }
    ]
})

# --------------------------------------------------
# 9. Print result
# --------------------------------------------------
print(result["messages"][-1].content)

The findings look valid for this sample dataset. The highest delay by gender is 34.2 days for males, and the highest delay by region is 40.0 days for Leeds.


In [ ]:
# Step 5: Create Final NHS Cancer Report

# Install required packages
!pip install -q langchain langchain-groq pandas

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
import pandas as pd
import os

# --------------------------------------------------
# 1. Create the LLM
# --------------------------------------------------
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="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",
    temperature=0
)

# --------------------------------------------------
# 2. File names
# --------------------------------------------------
raw_file = "nhs_cancer_data.csv"
cleaned_file = "cleaned_nhs_cancer_data.csv"
report_file = "final_nhs_cancer_report.txt"

# --------------------------------------------------
# 3. Create sample raw CSV if missing
# --------------------------------------------------
if not os.path.exists(raw_file):
    sample_data = {
        "patient_id": [1, 2, 3, 4, 5, 6, 7, 8],
        "age": [45, 67, 52, 71, 39, 58, 64, 50],
        "gender": ["Female", "Male", "Female", "Male", "Female", "Male", "Female", "Male"],
        "ethnicity": [
            "White", "Asian", "Black", "White",
            "Asian", "Black", "White", "Asian"
        ],
        "region": [
            "London", "Manchester", "Birmingham", "Leeds",
            "London", "Liverpool", "Bristol", "Manchester"
        ],
        "cancer_type": [
            "Breast", "Lung", "Breast", "Prostate",
            "Cervical", "Lung", "Breast", "Bowel"
        ],
        "referral_date": [
            "2024-01-10", "2024-02-12", "2024-01-25", "2024-03-01",
            "2024-02-15", "2024-01-18", "2024-02-05", "2024-03-10"
        ],
        "diagnosis_date": [
            "2024-02-05", "2024-03-20", "2024-02-15", "2024-04-10",
            "2024-03-01", "2024-02-25", "2024-03-10", "2024-04-01"
        ]
    }

    sample_df = pd.DataFrame(sample_data)
    sample_df.to_csv(raw_file, index=False)
    print(f"Created sample file: {raw_file}")

# --------------------------------------------------
# 4. Create cleaned CSV if missing
# --------------------------------------------------
if not os.path.exists(cleaned_file):
    df = pd.read_csv(raw_file)

    # Clean column names
    df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]

    # Keep useful columns
    useful_columns = [
        "patient_id",
        "age",
        "gender",
        "ethnicity",
        "region",
        "cancer_type",
        "diagnosis_date",
        "referral_date"
    ]
    df = df[useful_columns]

    # Remove duplicates
    df = df.drop_duplicates()

    # Remove rows with missing patient_id
    df = df.dropna(subset=["patient_id"])

    # Convert date columns
    df["diagnosis_date"] = pd.to_datetime(df["diagnosis_date"])
    df["referral_date"] = pd.to_datetime(df["referral_date"])

    # Save cleaned file
    df.to_csv(cleaned_file, index=False)
    print(f"Created cleaned file: {cleaned_file}")

# --------------------------------------------------
# 5. Tool: create final report
# --------------------------------------------------
@tool
def create_final_report() -> str:
    """Create the final NHS cancer data report and save it to a text file."""

    df = pd.read_csv(cleaned_file)

    # Convert dates
    df["diagnosis_date"] = pd.to_datetime(df["diagnosis_date"])
    df["referral_date"] = pd.to_datetime(df["referral_date"])

    # Create delay column
    df["delay_days"] = (df["diagnosis_date"] - df["referral_date"]).dt.days

    # Calculate key values
    total_rows = len(df)
    avg_delay = round(df["delay_days"].mean(), 1)

    gender_delay = df.groupby("gender")["delay_days"].mean().sort_values(ascending=False)
    highest_gender = gender_delay.index[0]
    highest_gender_delay = round(gender_delay.iloc[0], 1)

    region_delay = df.groupby("region")["delay_days"].mean().sort_values(ascending=False)
    highest_region = region_delay.index[0]
    highest_region_delay = round(region_delay.iloc[0], 1)

    # Create final report text
    report_text = f"""
NHS Cancer Data Final Report

1. Total patients analysed: {total_rows}
2. Overall average diagnosis delay: {avg_delay} days
3. Gender with highest average delay: {highest_gender} ({highest_gender_delay} days)
4. Region with highest average delay: {highest_region} ({highest_region_delay} days)

Conclusion:
The sample NHS cancer dataset shows that {highest_gender} patients had the highest
average diagnosis delay. The region with the highest average delay was {highest_region}.
This result can help NHS teams identify where further review and improvement may be needed.
"""

    # Save report
    with open(report_file, "w") as f:
        f.write(report_text.strip())

    return (
        f"Final report created successfully.\n"
        f"Saved file: {report_file}\n\n"
        f"{report_text.strip()}"
    )

# --------------------------------------------------
# 6. Put tools in a list
# --------------------------------------------------
tools = [create_final_report]

# --------------------------------------------------
# 7. Create the agent
# --------------------------------------------------
agent = create_agent(
    model=llm,
    tools=tools
)

# --------------------------------------------------
# 8. Run Step 5
# --------------------------------------------------
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Create the final NHS cancer report and tell me the saved file name "
                "and the final report content."
            )
        }
    ]
})

# --------------------------------------------------
# 9. Print result
# --------------------------------------------------
print(result["messages"][-1].content)

The final NHS cancer report has been created and saved to a file named 'final_nhs_cancer_report.txt'. The report contains the total number of patients analyzed, the overall average diagnosis delay, the gender with the highest average delay, and the region with the highest average delay. The report concludes that Male patients had the highest average diagnosis delay and the region with the highest average delay was Leeds, suggesting that further review and improvement may be needed in these areas.
